In [1]:
import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'

In [2]:
%load_ext autoreload
%autoreload 2

import sys; sys.path.append('..')
import MeshFEM, mesh, elastic_sheet, elastic_solid, energy, tensors, benchmark
import meshing, triangulation, py_newton_optimizer
from io_redirection import suppress_stdout as so
import sheet_convergence, sim_utils, semisphere_convergence
from tri_mesh_viewer import TriMeshViewer
import numpy as np, time
import copy

In [3]:
# when thickness is small try large force
thickness = 0.01
mV = 2e-5
tm = semisphere_convergence.getRecTetMesh(thickness,maxVol=mV)
esolid = semisphere_convergence.getElasticSolid(tm)

In [4]:
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.niter = 200
opts.gradTol = 1e-12

In [5]:
benchmark.reset()
esolid_sim,t1 = semisphere_convergence.cantilverGraSimulation(esolid, opts=opts)
benchmark.report()
esolid_energy = esolid_sim.energy()
print(esolid_sim.energy())

0	1.0201e-19	1.27235e-09	0.000244141	0
1	-1.65693e-08	4.25989e-06	0.0625	0
2	-2.73254e-08	6.96829e-06	0.5	0
3	-4.96482e-08	8.9465e-06	1	0
4	-9.51188e-08	5.81431e-06	0.25	0
5	-1.08113e-07	7.76118e-06	1	0
6	-1.45244e-07	7.60192e-06	0.25	0
7	-1.61551e-07	7.14502e-06	1	0
8	-1.94897e-07	7.60031e-06	0.5	0
9	-2.18265e-07	7.83441e-06	1	0
10	-2.57921e-07	5.30725e-06	0.5	0
11	-2.70072e-07	8.66522e-06	1	0
12	-3.12523e-07	6.16693e-06	0.5	0
13	-3.18785e-07	9.37041e-06	1	0
14	-3.63656e-07	6.49747e-06	0.5	0
15	-3.72047e-07	8.49624e-06	1	0
16	-4.11573e-07	6.30459e-06	0.5	0
17	-4.23381e-07	7.07323e-06	1	0
18	-4.55578e-07	6.15542e-06	0.5	0
19	-4.69585e-07	5.68828e-06	1	0
20	-4.95712e-07	6.02592e-06	0.5	0
21	-5.10263e-07	4.61237e-06	1	0
22	-5.3179e-07	5.94644e-06	1	0
23	-5.45089e-07	6.95307e-06	0.5	0
24	-5.66542e-07	5.89894e-06	1	0
25	-5.89438e-07	3.42611e-06	0.5	0
26	-5.98982e-07	4.96736e-06	1	0
27	-6.16836e-07	4.82422e-06	0.25	0
28	-6.23377e-07	4.32394e-06	1	0
29	-6.38912e-07	3.61202e-06	0.5	0
30	-6.48

In [6]:
import tri_mesh_viewer
tet_viewer = tri_mesh_viewer.Viewer(esolid_sim, wireframe=True)
tet_viewer.show()

Renderer(camera=PerspectiveCamera(children=(PointLight(color='#999999', position=(0.0, 0.0, 5.0), quaternion=(…

In [7]:
opts.factorizer = opts.factorizer.CatamariNesdis

In [8]:
opts.factorizer = opts.factorizer.CHOLMOD

In [9]:
mA = 1e-4
m = semisphere_convergence.getRecSheetMesh(maxArea=mA)
esheet = semisphere_convergence.getElasticSheet(m,thickness,useCreases=False)

In [10]:
benchmark.reset()
esheet_sim, t2 = semisphere_convergence.cantilverGraSimulation(esheet, opts=opts)
print(esheet_sim.energy(), esheet_sim.energy(etype=esheet_sim.EnergyType.Membrane), esheet_sim.energy(etype=esheet_sim.EnergyType.Bending))
benchmark.report()
esheet_energy = esheet_sim.energy()

0	1.43084e-20	2.40589e-09	0.000488281	0
1	-2.98105e-08	5.39827e-07	0.0625	0
2	-5.30713e-08	5.67965e-07	0.5	0
3	-9.26644e-08	6.4732e-07	1	0
4	-1.66941e-07	4.88612e-07	0.25	0
5	-1.94977e-07	5.57917e-07	1	0
6	-2.58061e-07	5.07556e-07	0.5	0
7	-2.96513e-07	5.53908e-07	1	0
8	-3.55114e-07	4.71005e-07	0.5	0
9	-3.85455e-07	5.30652e-07	1	0
10	-4.36774e-07	4.82351e-07	0.5	0
11	-4.65205e-07	4.62924e-07	1	0
12	-5.11566e-07	4.04e-07	0.5	0
13	-5.36383e-07	3.23012e-07	1	0
14	-5.71157e-07	4.1199e-07	1	0
15	-5.99409e-07	4.65255e-07	0.5	0
16	-6.25941e-07	3.77254e-07	1	0
17	-6.55172e-07	2.26698e-07	0.5	0
18	-6.68872e-07	3.65576e-07	1	0
19	-6.88777e-07	2.54266e-07	0.25	0
20	-6.94662e-07	3.95105e-07	1	0
21	-7.10119e-07	2.23532e-07	0.5	0
22	-7.13173e-07	9.44364e-07	1	0
23	-7.30165e-07	1.96397e-07	0.125	0
24	-7.31214e-07	4.63746e-07	1	0
25	-7.41231e-07	1.34235e-07	0.5	0
26	-7.45469e-07	3.28311e-07	1	0
27	-7.51609e-07	1.23146e-07	0.5	0
28	-7.54189e-07	2.34063e-07	1	0
29	-7.57984e-07	1.07409e-07	1	0
30	-7.58093

In [11]:
print(esheet_energy)

3.089356249393608e-08


In [12]:
es_viewer = tri_mesh_viewer.Viewer(esheet_sim, wireframe=True)
es_viewer.show()

Renderer(camera=PerspectiveCamera(children=(PointLight(color='#999999', position=(0.0, 0.0, 5.0), quaternion=(…

In [13]:
# Compute the vertex-averaged maximum principal strains
import field_sampler
fs = field_sampler.FieldSampler(esolid_sim.mesh())

emax_shell = [np.linalg.eigh(e)[0].max() for e in esheet_sim.vertexGreenStrains()]
emax_solid = [np.linalg.eigh(e)[0].max() for e in esolid_sim.vertexGreenStrains()]

# Also sample the tet simulation's piecewise linear vertex-averaged strain field at the midsurface sheet mesh vertices.
emax_solid_on_sheet = [np.linalg.eigh(e)[0].max() for e in fs.sample(esheet_sim.mesh().vertices(),
                                                                     np.array(esolid_sim.vertexGreenStrains()).reshape((-1, 9))).reshape(-1, 3, 3)]

In [44]:
# Compute the vertex-averaged maximum principal strains
import field_sampler
fs = field_sampler.FieldSampler(esolid_sim.mesh())

emax_shell = [np.linalg.eigh(e)[0].max() for e in esheet_sim.getVertexVolumetricStrains(thickness / 2)]
emax_solid = [np.linalg.eigh(e)[0].max() for e in esolid_sim.vertexGreenStrains()]

# Also sample the tet simulation's discontinuous piecewise linear strain field
# on the top surface using the sheet simulation mesh.
topSurfaceStrains = [esolid_sim.greenStrain(ei, bc) for ei, bc in zip(*fs.closestElementAndBaryCoords(esheet_sim.mesh().vertices() + np.array([0, 0, thickness / 2])))]
emax_solid_on_sheet = [np.linalg.eigh(e)[0].max() for e in topSurfaceStrains]
#emax_solid_on_sheet = fs.sample(esheet_sim.mesh().vertices() + np.array([0, 0, thickness / 2]), emax_solid)

In [45]:
np.mean(np.array(topSurfaceStrains))
# rel_err_energy = np.abs(esheet_sim.energy()-esolid_sim.energy())/esolid_sim.energy() 
# print(str(rel_err_energy*100) + '%')

8.158824211400244e-05

In [46]:
# Visualize strain and inspect mesh resolution on the undeformed configuration
v3 = TriMeshViewer(esheet_sim.mesh(), scalarField=emax_solid_on_sheet, wireframe=False)
v3.show()

Renderer(camera=PerspectiveCamera(children=(PointLight(color='#999999', position=(0.0, 0.0, 5.0), quaternion=(…

In [47]:
import vis
from ipywidgets import HBox

vmin = min(np.min(emax_shell), np.min(emax_solid_on_sheet))
vmax = max(np.max(emax_shell), np.max(emax_solid_on_sheet))

sf_shell1 = vis.fields.ScalarField(esheet_sim, emax_shell         , vmin=vmin, vmax=vmax)
sf_shell2 = vis.fields.ScalarField(esheet_sim, emax_solid_on_sheet, vmin=vmin, vmax=vmax)
sf_solid  = vis.fields.ScalarField(esolid_sim, emax_solid,          vmin=vmin, vmax=vmax)

vdefo = tri_mesh_viewer.Viewer(esolid_sim, scalarField=sf_solid) #volumetric deformation
sheetView = TriMeshViewer(esheet_sim, scalarField=sf_shell1)
sampledSolidView = TriMeshViewer(esheet_sim, scalarField=sf_shell2)
HBox([sheetView.show(), sampledSolidView.show(), vdefo.show()])

In [48]:
sampledSolidView.update(scalarField={'data': np.abs(sf_shell2.data - sf_shell1.data), 'vmin': vmin, 'vmax': vmax})

In [49]:
 np.abs(sf_shell2.data - sf_shell1.data).mean()

0.00010973478328057861

In [52]:
np.linalg.norm(np.array(emax_shell) - emax_solid_on_sheet) / np.linalg.norm(emax_shell)

0.08146295494427434

In [21]:
sheetView.showWireframe(True)

In [22]:
vdefo.tetShrinkFactor = 0.25

AttributeError: 'ScalarField' object has no attribute 'rawData'

In [ ]:
# # thicks_range = np.linspace(5e-3,0.2,30)
# res_set = semisphere_convergence.thickConvergenceSweep(semisphere_convergence.getRecTetMesh,semisphere_convergence.getRecSheetMesh,semisphere_convergence.cantilverGraSimulation)